# 02 · Whiplash index and April 1 snowpack

Full-record series per region, every winter 1950-51 onward, El Niño winters
highlighted (`ENSO_INDEX` from `src/config.py`). Descriptive only: no tests
before the freeze, and no fire data.

Inputs are built by code before this notebook runs (both cached in
`data/processed/`):

- `python -m src.whiplash`: Thornthwaite PET and SPEI-3 per ERA5-Land cell,
  log-logistic fit over 1951-2020, clipped to ±3.09, cos(latitude) regional
  means re-standardised over 1951-2020; whiplash = SPEI-3(JJA) - SPEI-3(DJF).
- `python -m src.snow`: April 1 00:00 UTC SWE from hourly ERA5-Land, glacier
  and perennial-snow cells excluded.
- `src.phase2`: the joined region x year table, written below to
  `data/processed/` and `results/tables/`.

Settings and their measured effects: `docs/methods_register.md`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd

from src.config import ENSO_INDEX, RESULTS
from src.describe import phase_summary
from src.figures import FIGURES, winter_series
from src.phase2 import SWE, WHIPLASH, build_table, write_tables

table = build_table(pd.read_csv(WHIPLASH), pd.read_csv(SWE))
write_tables(table)  # data/processed/ and results/tables/
print(f"{len(table)} rows, years {table['year'].min()}-{table['year'].max()}, index {ENSO_INDEX}")
table.head()

wrote C:\Users\dacek\Documents\Portfolio\hydrowhiplash-elnino26\data\processed\whiplash_swe_region_year.csv: 532 rows, 7 regions, years 1951-2026
wrote C:\Users\dacek\Documents\Portfolio\hydrowhiplash-elnino26\results\tables\whiplash_swe_region_year.csv: 532 rows, 7 regions, years 1951-2026
532 rows, years 1951-2026, index ONI


,region,year,winter,whiplash,spei3_djf,spei3_jja,swe_apr1_mm,spei3_djf_cellmean,spei3_jja_cellmean,enso_index,enso_djf,phase,strong,swe_apr1_mm_all_cells
0,PNW,1951,1950-1951,-1.695495,1.070277,-0.625218,210.869193,0.913767,-0.505059,ONI,-0.66,la_nina,False,210.869193
1,PNW,1952,1951-1952,-0.392559,0.739893,0.347335,270.312784,0.631586,0.268110,ONI,0.52,el_nino,False,270.312784
2,PNW,1953,1952-1953,-1.116627,1.834433,0.717806,186.764059,1.566432,0.562631,ONI,0.25,neutral,False,186.764059
3,PNW,1954,1953-1954,0.546044,0.935170,1.481214,210.690096,0.798372,1.169532,ONI,0.41,neutral,False,210.690096
4,PNW,1955,1954-1955,0.753916,-0.893164,-0.139248,174.135257,-0.763207,-0.118718,ONI,-0.50,la_nina,False,174.135257


## Glacier exclusion

Mean April 1 SWE per region with glacier and perennial-snow cells excluded,
against all cells. The excluded cells add depth without year-to-year change,
so they inflate the mean while barely touching the variability.

In [2]:
glacier = table.groupby("region", sort=False)[["swe_apr1_mm", "swe_apr1_mm_all_cells"]].agg(["mean", "std"])
glacier.round(1)

swe_apr1_mm       swe_apr1_mm_all_cells      
              mean   std                  mean   std
region                                              
PNW          135.8  56.0                 135.8  56.0
BC           442.0  71.6                 592.8  73.1
AB            67.1  27.4                  69.3  27.4
AK           225.8  32.3                 711.4  32.6
YT           167.2  28.6                 474.8  27.8
NorCal        68.0  48.5                  68.0  48.5
SoCal          0.1   0.2                   0.1   0.2

## Whiplash, every year

Year Y pairs winter (Y-1)-Y with summer Y. Negative = a swing from a wetter
than usual winter to a drier than usual summer. 2026 has a winter value but
no summer yet (ERA5-Land ends July 2026).

In [3]:
winter_series(
    table.dropna(subset=["whiplash"]), "whiplash", "SPEI-3 JJA minus DJF (SD)", ENSO_INDEX,
    FIGURES / f"whiplash_series_{ENSO_INDEX.lower()}.png", signed=True,
    xlabel="Winter (Dec-Feb) and the following summer (Jun-Aug); year Y = winter (Y-1)-Y and summer Y")

WindowsPath('C:/Users/dacek/Documents/Portfolio/hydrowhiplash-elnino26/results/figures/whiplash_series_oni.png')

## April 1 SWE, every winter

SoCal is left out of this figure: its regional April 1 SWE is close to zero
in every winter. It stays in the table.

In [4]:
winter_series(
    table[table["region"] != "SoCal"].dropna(subset=["swe_apr1_mm"]), "swe_apr1_mm",
    "April 1 SWE (mm)", ENSO_INDEX, FIGURES / f"swe_april1_series_{ENSO_INDEX.lower()}.png")

WindowsPath('C:/Users/dacek/Documents/Portfolio/hydrowhiplash-elnino26/results/figures/swe_april1_series_oni.png')

## Descriptive phase summary

Group means and El Niño differences, in the variable's units, as % of the
all-winter mean and in SDs of all winters. No p-values: those come after the
freeze.

In [5]:
cols = ["region", "n_el_nino", "n_strong", "n_neutral", "mean_all", "mean_el_nino", "mean_neutral",
        "nino_minus_neutral_pct", "nino_minus_neutral_sd", "strong_minus_all_pct",
        "strong_minus_all_sd", "nino_share_below_median"]
phase_summary(table[table["region"] != "SoCal"].dropna(subset=["swe_apr1_mm"]), "swe_apr1_mm")[cols].round(2)

,region,n_el_nino,n_strong,n_neutral,mean_all,mean_el_nino,mean_neutral,nino_minus_neutral_pct,nino_minus_neutral_sd,strong_minus_all_pct,strong_minus_all_sd,nino_share_below_median
0,PNW,25,7,25,135.80,115.47,124.79,-6.86,-0.17,-22.03,-0.53,0.72
1,BC,25,7,25,441.98,413.12,443.53,-6.88,-0.42,-12.42,-0.77,0.68
2,AB,25,7,25,67.10,54.73,70.11,-22.92,-0.56,-23.89,-0.58,0.72
3,AK,25,7,25,225.78,220.86,222.75,-0.84,-0.06,-5.41,-0.38,0.56
4,YT,25,7,25,167.22,161.04,172.74,-7.00,-0.41,-1.20,-0.07,0.60
5,NorCal,25,7,25,67.96,77.44,56.61,30.65,0.43,33.99,0.48,0.44


In [6]:
wcols = ["region", "n_el_nino", "n_strong", "mean_el_nino", "mean_neutral", "mean_strong",
         "nino_minus_neutral", "nino_minus_neutral_sd", "strong_minus_all_sd"]
w = table.dropna(subset=["whiplash"])
pd.concat({v: phase_summary(w, v)[wcols].set_index("region")
           for v in ["spei3_djf", "spei3_jja", "whiplash"]}, axis=1).round(2)

spei3_djf                                                 \
       n_el_nino n_strong mean_el_nino mean_neutral mean_strong   
region                                                            
PNW           25        7        -0.28         0.00       -0.16   
BC            25        7        -0.30         0.07       -0.30   
AB            25        7        -0.34         0.13       -0.70   
AK            25        7        -0.28         0.01       -1.19   
YT            25        7        -0.29         0.17       -0.37   
NorCal        25        7         0.23        -0.09        0.56   
SoCal         25        7         0.50        -0.24        0.66   

                                                                    spei3_jja  \
       nino_minus_neutral nino_minus_neutral_sd strong_minus_all_sd n_el_nino   
region                                                                          
PNW                 -0.28                 -0.27               -0.10        25   
BC                  -0.37                 -0.34               -0.20        25   
AB                  -0.47                 -0.42               -0.59        25   
AK                  -0.29                 -0.26               -1.09        25   
YT                  -0.46                 -0.44               -0.36        25   
NorCal               0.31                  0.32                0.62        25   
SoCal                0.74                  0.75                0.70        25   

                 ...                                            whiplash  \
       n_strong  ... nino_minus_neutral_sd strong_minus_all_sd n_el_nino   
region           ...                                                       
PNW           7  ...                  0.12                0.07        25   
BC            7  ...                  0.36               -0.37        25   
AB            7  ...                  0.42                0.08        25   
AK            7  ...                  0.00                0.76        25   
YT            7  ...                  0.05               -0.21        25   
NorCal        7  ...                  0.21                0.10        25   
SoCal         7  ...                 -0.07               -0.16        25   

                                                                          \
       n_strong mean_el_nino mean_neutral mean_strong nino_minus_neutral   
region                                                                     
PNW           7         0.28        -0.13        0.12               0.41   
BC            7         0.49        -0.27       -0.17               0.75   
AB            7         0.58        -0.34        0.70               0.92   
AK            7         0.32         0.03        1.97               0.29   
YT            7         0.29        -0.24        0.06               0.52   
NorCal        7        -0.18        -0.09       -0.56              -0.09   
SoCal         7        -0.63         0.18       -0.92              -0.81   

                                                  
       nino_minus_neutral_sd strong_minus_all_sd  
region                                            
PNW                     0.29                0.12  
BC                      0.53               -0.12  
AB                      0.62                0.50  
AK                      0.18                1.24  
YT                      0.37                0.11  
NorCal                 -0.07               -0.38  
SoCal                  -0.57               -0.61  

[7 rows x 24 columns]